In [1]:
import polars as pl
import os

# 1. 파일 경로 설정
file_path = "/home/tracerofjageum/graph_structure_only.parquet"

# 2. 파일 존재 여부 및 크기 확인
if os.path.exists(file_path):
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"✅ 파일이 존재합니다: {file_path}")
    print(f"📦 파일 크기: {file_size_mb:.2f} MB")
    
    if file_size_mb < 0.01: # 10KB 미만이면 비정상으로 간주
        print("⚠️ 경고: 파일 크기가 너무 작습니다. 데이터가 비어있을 수 있습니다.")
else:
    print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
    # 파일이 없으면 더 이상 진행하지 않음
    exit()

print("-" * 50)

# 3. 데이터 로드 및 무결성 검사
try:
    print("📂 데이터 로딩 중...")
    df_graph = pl.read_parquet(file_path)
    
    # 3-1. 기본 정보 출력
    print(f"✅ 데이터 로드 성공!")
    print(f"📊 행(Row) 수: {df_graph.height}")
    print(f"📊 열(Column) 수: {df_graph.width}")
    print(f"📋 컬럼 목록: {df_graph.columns}")
    
    print("-" * 50)
    
    # 3-2. 결측치(Null) 확인
    print("🔍 결측치(Null) 점검:")
    null_counts = df_graph.null_count()
    for col in df_graph.columns:
        n_null = null_counts[col][0]
        percent = (n_null / df_graph.height) * 100
        print(f"   - {col}: {n_null}개 ({percent:.2f}%)")
        
    print("-" * 50)

    # 3-3. 기초 통계량 확인 (수치형 컬럼만)
    print("📈 기초 통계량 (Describe):")
    # account_id나 community_id 같은 식별자는 제외하고 통계량 확인
    numeric_cols = [col for col in df_graph.columns if col not in ['account_id', 'community_id']]
    
    if numeric_cols:
        print(df_graph.select(numeric_cols).describe())
    else:
        print("⚠️ 수치형 피처가 감지되지 않았습니다.")

    print("-" * 50)
    
    # 3-4. 샘플 데이터 출력
    print("👀 데이터 샘플 (상위 5행):")
    print(df_graph.head(5))

except Exception as e:
    print(f"❌ 데이터 로드 중 에러 발생: {e}")

❌ 파일을 찾을 수 없습니다: /home/tracerofjageum/graph_structure_only.parquet
--------------------------------------------------
📂 데이터 로딩 중...
❌ 데이터 로드 중 에러 발생: No such file or directory (os error 2): /home/tracerofjageum/graph_structure_only.parquet


In [1]:
import polars as pl
import os

# 1. 파일 찾기 함수 (재귀 검색)
def find_file(filename, search_path="."):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

target_filename = "graph_structure_only.parquet"
print(f"🕵️ '{target_filename}' 파일을 찾는 중...")

# 현재 디렉토리부터 검색 시작
found_path = find_file(target_filename)

if found_path:
    print(f"✅ 파일을 찾았습니다! 경로: {found_path}")
    print("-" * 50)
    
    # 2. 무결성 검사 시작
    try:
        # 파일 크기 확인
        file_size_mb = os.path.getsize(found_path) / (1024 * 1024)
        print(f"📦 파일 크기: {file_size_mb:.2f} MB")
        
        if file_size_mb < 0.01:
            print("⚠️ 경고: 파일 크기가 너무 작습니다 (비어있음).")
        
        # 데이터 로드
        print("📂 데이터 로딩 중...")
        df_graph = pl.read_parquet(found_path)
        
        print(f"✅ 데이터 로드 성공!")
        print(f"📊 행(Row) 수: {df_graph.height}")
        print(f"📊 열(Column) 수: {df_graph.width}")
        print(f"📋 컬럼 목록: {df_graph.columns}")
        
        print("-" * 50)
        
        # 결측치 확인
        print("🔍 결측치(Null) 점검:")
        null_counts = df_graph.null_count()
        for col in df_graph.columns:
            n_null = null_counts[col][0]
            percent = (n_null / df_graph.height) * 100
            if n_null > 0:
                print(f"   - {col}: {n_null}개 ({percent:.2f}%)")
            else:
                print(f"   - {col}: 0개 (Clean ✨)")
                
        print("-" * 50)

        # 기초 통계량 (Describe)
        print("📈 기초 통계량 (Describe):")
        numeric_cols = [col for col in df_graph.columns if col not in ['account_id', 'community_id']]
        if numeric_cols:
            print(df_graph.select(numeric_cols).describe())
            
        print("-" * 50)
        
        # 샘플 출력
        print("👀 데이터 샘플 (상위 5행):")
        print(df_graph.head(5))
        
    except Exception as e:
        print(f"❌ 데이터 검사 중 에러 발생: {e}")

else:
    print("❌ 파일을 찾을 수 없습니다.")
    print("현재 폴더 내의 하위 디렉토리 목록:")
    try:
        print([d for d in os.listdir(".") if os.path.isdir(d)])
    except:
        print("디렉토리 목록을 읽을 수 없습니다.")

🕵️ 'graph_structure_only.parquet' 파일을 찾는 중...
✅ 파일을 찾았습니다! 경로: ./graph_structure_only.parquet
--------------------------------------------------
📦 파일 크기: 0.01 MB
⚠️ 경고: 파일 크기가 너무 작습니다 (비어있음).
📂 데이터 로딩 중...
✅ 데이터 로드 성공!
📊 행(Row) 수: 6
📊 열(Column) 수: 19
📋 컬럼 목록: ['account', 'is_fraud', 'pagerank', 'in_degree', 'out_degree', 'total_degree', 'clustering_coefficient', 'degree_centrality', 'in_degree_centrality', 'out_degree_centrality', 'betweenness_centrality', 'community_id', 'degree_ratio', 'degree_difference', 'aml_suspicion_score', 'community_size', 'community_fraud_ratio', 'num_neighbors', 'num_predecessors']
--------------------------------------------------
🔍 결측치(Null) 점검:
   - account: 0개 (Clean ✨)
   - is_fraud: 0개 (Clean ✨)
   - pagerank: 0개 (Clean ✨)
   - in_degree: 0개 (Clean ✨)
   - out_degree: 0개 (Clean ✨)
   - total_degree: 0개 (Clean ✨)
   - clustering_coefficient: 0개 (Clean ✨)
   - degree_centrality: 0개 (Clean ✨)
   - in_degree_centrality: 0개 (Clean ✨)
   - out_degree_central

In [1]:
import polars as pl
import networkx as nx
import pandas as pd
import numpy as np
import gc
import os

# =============================================================================
# 1. 설정 및 데이터 로드
# =============================================================================
input_path = '/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet'
output_path = '/home/tracerofjageum/my-lab/첫번째 모델/graph_features_final.parquet'

print("🚀 [그래프 피처 생성] 시작합니다...")
print(f"📂 Master 파일 로딩 중: {input_path}")

# 필요한 컬럼만 쏙 뽑아서 로드 (메모리 절약)
cols_to_use = ["from_acc", "to_acc", "Amount Paid", "Is Laundering"]
df = pl.read_parquet(input_path, columns=cols_to_use)

# 컬럼명 표준화 (NetworkX 처리를 위해)
df = df.rename({
    "from_acc": "source",
    "to_acc": "target",
    "Amount Paid": "amount",
    "Is Laundering": "is_laundering"
})

# 금액 결측치 0 처리
df = df.with_columns(pl.col("amount").fill_null(0.0))

print(f"✅ 데이터 로드 완료! (행 수: {df.height:,})")

🚀 [그래프 피처 생성] 시작합니다...
📂 Master 파일 로딩 중: /home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet
✅ 데이터 로드 완료! (행 수: 31,898,669)


In [1]:
import polars as pl
import pandas as pd
import numpy as np
import os
import gc
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# 1. 라이브러리 및 경로 설정
# -----------------------------------------------------------------------------
try:
    import cugraph
    import cudf
    print("✅ cuGraph (GPU) 활성화")
except ImportError:
    raise ImportError("❌ GPU 환경에서 cuGraph가 필요합니다.")

try:
    import networkit as nk
    print("✅ NetworKit 활성화")
except ImportError:
    print("⚠️ NetworKit 없음. 일부 기능이 제한될 수 있습니다.")

input_path = "/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet"
output_path = "./graph_features_final_v3.parquet"

# -----------------------------------------------------------------------------
# 2. 데이터 로드 및 시간 기반 분할 (가이드라인 준수)
# -----------------------------------------------------------------------------
print(f"📂 데이터 로딩 중...")
df_all = pl.read_parquet(input_path)

# 가이드라인: 시점 기준으로 분리
if "Timestamp" in df_all.columns:
    df_all = df_all.sort("Timestamp")
    print("✅ Timestamp 기준 정렬 완료")

total_rows = len(df_all)
train_size = int(total_rows * 0.6) # 60% 학습셋

# 학습셋 슬라이싱 (그래프 구축용)
df_train = df_all.slice(0, train_size)

# -----------------------------------------------------------------------------
# 3. 사기 계좌 정의 (★PDF 가이드 수정 반영★)
# 가이드: Is Laundering이 1인 경우 from/to account 둘 다 자금세탁 계좌로 정의
# -----------------------------------------------------------------------------
print("🔍 [수정] 가이드라인에 따른 사기 계좌(From/To 모두) 추출 중...")

fraud_from = (df_train.filter(pl.col("Is Laundering") == 1)
              .select(pl.col("from_acc").alias("account")).unique())
fraud_to = (df_train.filter(pl.col("Is Laundering") == 1)
            .select(pl.col("to_acc").alias("account")).unique())

# From과 To를 합쳐서 고유한 사기 계좌 리스트 생성
fraud_accounts_train = pl.concat([fraud_from, fraud_to]).unique().to_series().to_list()
fraud_set_train = set(fraud_accounts_train)

print(f"✅ 학습셋 내 사기 연루 계좌 수: {len(fraud_set_train):,}개")

# -----------------------------------------------------------------------------
# 4. 노드 매핑 및 엣지 집계 (Polars 최적화)
# -----------------------------------------------------------------------------
print("\n🚀 노드 매핑 및 엣지 집계 시작")

# 전체 기간 등장하는 모든 계좌 (Test 데이터에 피처를 붙이기 위해 필요)
all_nodes = pl.concat([
    df_all.select(pl.col("from_acc").alias("account")),
    df_all.select(pl.col("to_acc").alias("account"))
]).unique()

try:
    all_nodes = all_nodes.with_row_index("node_id")
except AttributeError:
    all_nodes = all_nodes.with_row_count("node_id")

# Train 데이터 엣지 구성 (학습 기간의 관계만 사용!)
df_train_mapped = (
    df_train
    .join(all_nodes, left_on="from_acc", right_on="account", how="left").rename({"node_id": "src"})
    .join(all_nodes, left_on="to_acc", right_on="account", how="left").rename({"node_id": "dst"})
    .select(["src", "dst", "Amount Paid"])
)

# 엣지 집계
edge_agg_pl = df_train_mapped.group_by(["src", "dst"]).agg([
    pl.col("Amount Paid").sum().alias("weight")
])
edge_agg_pd = edge_agg_pl.to_pandas()

del df_train_mapped
gc.collect()

# -----------------------------------------------------------------------------
# 5. cuGraph & NetworKit 기반 피처 계산
# -----------------------------------------------------------------------------
print("\n🕸️ 그래프 알고리즘 계산 중...")

# GPU 데이터 변환
cu_edges = cudf.DataFrame({
    'src': edge_agg_pd['src'].astype('int32').values,
    'dst': edge_agg_pd['dst'].astype('int32').values,
    'weight': edge_agg_pd['weight'].astype('float32').values
})

# 그래프 생성
G_directed = cugraph.Graph(directed=True)
G_directed.from_cudf_edgelist(cu_edges, source="src", destination="dst", edge_attr="weight", renumber=False)
G_undirected = cugraph.Graph(directed=False)
G_undirected.from_cudf_edgelist(cu_edges, source="src", destination="dst", edge_attr="weight", renumber=False)

# 피처 계산 (PageRank, Degree, Community)
pagerank_df = cugraph.pagerank(G_directed)
pagerank_dict = dict(zip(pagerank_df['vertex'].to_pandas(), pagerank_df['pagerank'].to_pandas()))

in_degree_df = G_directed.in_degree()
out_degree_df = G_directed.out_degree()
in_degree_dict = dict(zip(in_degree_df['vertex'].to_pandas(), in_degree_df['degree'].to_pandas()))
out_degree_dict = dict(zip(out_degree_df['vertex'].to_pandas(), out_degree_df['degree'].to_pandas()))

community_df, _ = cugraph.louvain(G_undirected)
community_dict = dict(zip(community_df['vertex'].to_pandas(), community_df['partition'].to_pandas()))

# NetworKit Betweenness
num_nodes_total = len(all_nodes)
G_nk = nk.Graph(n=num_nodes_total, weighted=True, directed=False)
for row in tqdm(edge_agg_pd.itertuples(index=False), total=len(edge_agg_pd), desc="Graph Build"):
    G_nk.addEdge(int(row.src), int(row.dst), w=float(row.weight))

approx_bc = nk.centrality.ApproxBetweenness(G_nk, epsilon=0.1, delta=0.1)
approx_bc.run()
betweenness_dict = {i: approx_bc.score(i) for i in range(num_nodes_total)}

# -----------------------------------------------------------------------------
# 6. 결과 매핑 및 가이드라인 준수형 파생 피처 생성
# -----------------------------------------------------------------------------
print("\n💾 결과 통합 및 저장 중...")

id_to_node = dict(zip(all_nodes["node_id"], all_nodes["account"]))
graph_features = []

for node_id in tqdm(range(num_nodes_total), desc="Finalizing"):
    original_account = id_to_node.get(node_id)
    if not original_account: continue
    
    feat = {
        'account_id': original_account,
        'pagerank': pagerank_dict.get(node_id, 0.0),
        'in_degree': in_degree_dict.get(node_id, 0),
        'out_degree': out_degree_dict.get(node_id, 0),
        'community_id': community_dict.get(node_id, -1),
        'betweenness_centrality': betweenness_dict.get(node_id, 0.0)
    }
    graph_features.append(feat)

df_final = pl.DataFrame(graph_features)

# 파생 피처 (AML 의심 점수화)
df_final = df_final.with_columns([
    (pl.col("in_degree") / (pl.col("out_degree") + 1)).alias("degree_ratio"),
    (pl.col("account_id").is_in(list(fraud_set_train)).cast(pl.Int8)).alias("is_fraud_node_train")
])

# 커뮤니티 사기 비율 (학습셋 기준)
comm_fraud = df_final.filter(pl.col("community_id") != -1).group_by("community_id").agg(
    pl.col("is_fraud_node_train").mean().alias("community_fraud_ratio")
)
df_final = df_final.join(comm_fraud, on="community_id", how="left").fill_null(0)

# 최종 저장
df_final.write_parquet(output_path)

print("="*80)
print(f"🎉 [성공] 가이드라인 준수형 그래프 피처 생성 완료!")
print(f"   - 사기 계좌 정의: From & To 모두 포함")
print(f"   - 데이터 누수 방지: 학습 기간(60%) 관계만 사용")
print(f"   - 저장 위치: {output_path}")
print("="*80)

✅ cuGraph (GPU) 활성화


✅ NetworKit 활성화
📂 데이터 로딩 중...
✅ Timestamp 기준 정렬 완료
🔍 [수정] 가이드라인에 따른 사기 계좌(From/To 모두) 추출 중...
✅ 학습셋 내 사기 연루 계좌 수: 20,459개

🚀 노드 매핑 및 엣지 집계 시작

🕸️ 그래프 알고리즘 계산 중...


Graph Build: 100%|████████████████████████████████████████████████| 3991748/3991748 [00:08<00:00, 489245.17it/s]



💾 결과 통합 및 저장 중...


Finalizing: 100%|█████████████████████████████████████████████████| 2076999/2076999 [00:03<00:00, 582281.30it/s]


🎉 [성공] 가이드라인 준수형 그래프 피처 생성 완료!
   - 사기 계좌 정의: From & To 모두 포함
   - 데이터 누수 방지: 학습 기간(60%) 관계만 사용
   - 저장 위치: ./graph_features_final_v3.parquet


In [2]:
import polars as pl
import os

# 1. 파일 경로 설정 (방금 생성한 V3 파일로 변경)
file_path = "./graph_features_final_v3.parquet"

print(f"🕵️ [V3 가이드라인 준수형] 파일 무결성 검사 시작: {file_path}")

if not os.path.exists(file_path):
    print("❌ 파일을 찾을 수 없습니다. 이전 단계(생성)가 완료되었는지 확인해주세요.")
else:
    try:
        # 2. 데이터 로드
        df_graph = pl.read_parquet(file_path)
        
        print("\n" + "="*50)
        print("✅ 1. 기본 정보 (Basic Info)")
        print("="*50)
        print(f"📊 총 계좌(Node) 수: {df_graph.height:,}")
        print(f"📊 피처(Column) 수: {df_graph.width}")
        print(f"📋 컬럼 목록:\n{df_graph.columns}")

        # 3. 샘플 데이터 (Head)
        print("\n" + "="*50)
        print("👀 2. 데이터 미리보기 (Head 5)")
        print("="*50)
        # account -> account_id로 변경된 점 반영
        print(df_graph.head(5))

        # 4. 결측치 및 활성값 점검
        print("\n" + "="*50)
        print("🔍 3. 무결성 및 가이드라인 준수 여부 점검")
        print("="*50)
        
        # 가이드라인 핵심 피처 포함 확인
        numeric_cols = [
            "pagerank", 
            "betweenness_centrality", 
            "community_fraud_ratio", 
            "aml_suspicion_score", 
            "degree_ratio",
            "is_fraud_node_train" # 가이드라인(from/to 모두 사기) 반영 여부 확인용
        ]
        
        target_cols = [c for c in numeric_cols if c in df_graph.columns]
        
        if target_cols:
            # 주요 피처 통계 요약
            stats = df_graph.select(target_cols).describe()
            print(stats)
            
            print("\n[피처별 유효 데이터 분포 확인]")
            for col in target_cols:
                # 0보다 큰(활성화된) 데이터 수 계산
                active_count = df_graph.filter(pl.col(col) > 0).height
                print(f"   - {col:<25}: 활성값 {active_count:>8,}개 ({active_count/df_graph.height*100:5.1f}%)")
                
                if active_count == 0:
                    print(f"     ⚠️ 경고: {col} 피처의 모든 값이 0입니다. 로직을 재확인하세요.")

        # 5. 특이사항 점검 (가이드라인 준수 확인)
        print("\n[가이드라인 준수 포인트 확인]")
        
        # 사기 노드(Train 기준) 식별 수
        if "is_fraud_node_train" in df_graph.columns:
            fraud_node_count = df_graph.filter(pl.col("is_fraud_node_train") == 1).height
            print(f"   🚩 학습 데이터 내 사기 연루 계좌(From/To): {fraud_node_count:,} 개")
            if fraud_node_count == 0:
                print("     ⚠️ 경고: 학습셋 내 사기 계좌가 추출되지 않았습니다.")

        # 커뮤니티 탐지 상태
        if "community_id" in df_graph.columns:
            n_communities = df_graph.select(pl.col("community_id").n_unique()).item()
            print(f"   🏘️ 탐지된 고유 커뮤니티(집단) 수: {n_communities:,} 개")

    except Exception as e:
        print(f"❌ 데이터 검사 중 에러 발생: {e}")

🕵️ [V3 가이드라인 준수형] 파일 무결성 검사 시작: ./graph_features_final_v3.parquet

✅ 1. 기본 정보 (Basic Info)
📊 총 계좌(Node) 수: 2,076,999
📊 피처(Column) 수: 9
📋 컬럼 목록:
['account_id', 'pagerank', 'in_degree', 'out_degree', 'community_id', 'betweenness_centrality', 'degree_ratio', 'is_fraud_node_train', 'community_fraud_ratio']

👀 2. 데이터 미리보기 (Head 5)
shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ account_i ┆ pagerank  ┆ in_degree ┆ out_degre ┆ … ┆ betweenne ┆ degree_ra ┆ is_fraud_ ┆ communit │
│ d         ┆ ---       ┆ ---       ┆ e         ┆   ┆ ss_centra ┆ tio       ┆ node_trai ┆ y_fraud_ │
│ ---       ┆ f64       ┆ i64       ┆ ---       ┆   ┆ lity      ┆ ---       ┆ n         ┆ ratio    │
│ str       ┆           ┆           ┆ i64       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆           ┆ i8        ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══

In [2]:
import polars as pl
import pandas as pd
import numpy as np
import os
import gc
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# 1. 라이브러리 및 경로 설정
# -----------------------------------------------------------------------------
try:
    import cugraph
    import cudf
    print("✅ cuGraph (GPU) 활성화")
except ImportError:
    raise ImportError("❌ GPU 환경에서 cuGraph가 필요합니다.")

try:
    import networkit as nk
    print("✅ NetworKit 활성화")
except ImportError:
    print("⚠️ NetworKit 없음. 일부 기능이 제한될 수 있습니다.")

# 입력 경로 및 출력 경로 (누락 방지 버전 이름으로 변경)
input_path = "/home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet"
output_path = "./graph_features_v3_fixed_no_leakage.parquet"

# -----------------------------------------------------------------------------
# 2. 데이터 로드 및 시간 기반 분할
# -----------------------------------------------------------------------------
print(f"📂 데이터 로딩 중...")
df_all = pl.read_parquet(input_path)

if "Timestamp" in df_all.columns:
    df_all = df_all.sort("Timestamp")
    print("✅ Timestamp 기준 정렬 완료")

total_rows = len(df_all)
train_size = int(total_rows * 0.6)
df_train = df_all.slice(0, train_size)

# -----------------------------------------------------------------------------
# 3. 사기 계좌 정의 (From/To 모두 포함)
# -----------------------------------------------------------------------------
print("🔍 사기 계좌 추출 중...")
fraud_from = (df_train.filter(pl.col("Is Laundering") == 1)
              .select(pl.col("from_acc").alias("account")).unique())
fraud_to = (df_train.filter(pl.col("Is Laundering") == 1)
            .select(pl.col("to_acc").alias("account")).unique())

fraud_accounts_train = pl.concat([fraud_from, fraud_to]).unique().to_series().to_list()
fraud_set_train = set(fraud_accounts_train)

# -----------------------------------------------------------------------------
# 4. 노드 매핑 및 엣지 집계
# -----------------------------------------------------------------------------
all_nodes = pl.concat([
    df_all.select(pl.col("from_acc").alias("account")),
    df_all.select(pl.col("to_acc").alias("account"))
]).unique()

try:
    all_nodes = all_nodes.with_row_index("node_id")
except AttributeError:
    all_nodes = all_nodes.with_row_count("node_id")

df_train_mapped = (
    df_train
    .join(all_nodes, left_on="from_acc", right_on="account", how="left").rename({"node_id": "src"})
    .join(all_nodes, left_on="to_acc", right_on="account", how="left").rename({"node_id": "dst"})
    .select(["src", "dst", "Amount Paid"])
)

edge_agg_pl = df_train_mapped.group_by(["src", "dst"]).agg([
    pl.col("Amount Paid").sum().alias("weight")
])
edge_agg_pd = edge_agg_pl.to_pandas()
del df_train_mapped; gc.collect()

# -----------------------------------------------------------------------------
# 5. cuGraph & NetworKit 기반 피처 계산 (Clustering 복구)
# -----------------------------------------------------------------------------
print("\n🕸️ 그래프 알고리즘 계산 중...")
cu_edges = cudf.DataFrame({
    'src': edge_agg_pd['src'].astype('int32').values,
    'dst': edge_agg_pd['dst'].astype('int32').values,
    'weight': edge_agg_pd['weight'].astype('float32').values
})

G_directed = cugraph.Graph(directed=True)
G_directed.from_cudf_edgelist(cu_edges, source="src", destination="dst", edge_attr="weight", renumber=False)
G_undirected = cugraph.Graph(directed=False)
G_undirected.from_cudf_edgelist(cu_edges, source="src", destination="dst", edge_attr="weight", renumber=False)

# 1) PageRank
pagerank_df = cugraph.pagerank(G_directed)
pagerank_dict = dict(zip(pagerank_df['vertex'].to_pandas(), pagerank_df['pagerank'].to_pandas()))

# 2) Degree
in_degree_df = G_directed.in_degree()
out_degree_df = G_directed.out_degree()
in_degree_dict = dict(zip(in_degree_df['vertex'].to_pandas(), in_degree_df['degree'].to_pandas()))
out_degree_dict = dict(zip(out_degree_df['vertex'].to_pandas(), out_degree_df['degree'].to_pandas()))

# 3) Clustering Coefficient (★누락 복구 로직★)
print("📊 Clustering Coefficient 계산 중...")
clustering_df = cugraph.triangle_count(G_undirected)
clustering_dict = {}
for _, row in clustering_df.to_pandas().iterrows():
    v = int(row['vertex'])
    tri = int(row['counts'])
    deg = in_degree_dict.get(v, 0) + out_degree_dict.get(v, 0)
    # 군집 계수 공식: (2 * 삼각형 수) / (차수 * (차수-1))
    if deg >= 2:
        clustering_dict[v] = (2 * tri) / (deg * (deg - 1))
    else:
        clustering_dict[v] = 0.0

# 4) Louvain Community
community_df, _ = cugraph.louvain(G_undirected)
community_dict = dict(zip(community_df['vertex'].to_pandas(), community_df['partition'].to_pandas()))

# 5) NetworKit Betweenness
num_nodes_total = len(all_nodes)
G_nk = nk.Graph(n=num_nodes_total, weighted=True, directed=False)
for row in tqdm(edge_agg_pd.itertuples(index=False), total=len(edge_agg_pd), desc="NetworKit Build"):
    G_nk.addEdge(int(row.src), int(row.dst), w=float(row.weight))

approx_bc = nk.centrality.ApproxBetweenness(G_nk, epsilon=0.1, delta=0.1)
approx_bc.run()
betweenness_dict = {i: approx_bc.score(i) for i in range(num_nodes_total)}

# -----------------------------------------------------------------------------
# 6. 결과 매핑 (★Clustering 피처 딕셔너리 추가★)
# -----------------------------------------------------------------------------
print("\n💾 결과 통합 중...")
id_to_node = dict(zip(all_nodes["node_id"], all_nodes["account"]))
graph_features = []

for node_id in tqdm(range(num_nodes_total), desc="Finalizing"):
    original_account = id_to_node.get(node_id)
    if not original_account: continue
    
    feat = {
        'account_id': original_account,
        'pagerank': pagerank_dict.get(node_id, 0.0),
        'in_degree': in_degree_dict.get(node_id, 0),
        'out_degree': out_degree_dict.get(node_id, 0),
        'clustering_coefficient': clustering_dict.get(node_id, 0.0), # ★복구 완료
        'community_id': community_dict.get(node_id, -1),
        'betweenness_centrality': betweenness_dict.get(node_id, 0.0)
    }
    graph_features.append(feat)

df_final = pl.DataFrame(graph_features)

# -----------------------------------------------------------------------------
# 7. 파생 피처 생성 및 저장
# -----------------------------------------------------------------------------
df_final = df_final.with_columns([
    (pl.col("in_degree") / (pl.col("out_degree") + 1)).alias("degree_ratio"),
    # Clustering Coefficient가 복구되었으므로 Suspicion Score가 정상 계산됨
    (pl.col("pagerank") * pl.col("clustering_coefficient") * 1000).alias("aml_suspicion_score"),
    (pl.col("account_id").is_in(list(fraud_set_train)).cast(pl.Int8)).alias("is_fraud_node_train")
])

comm_fraud = df_final.filter(pl.col("community_id") != -1).group_by("community_id").agg(
    pl.col("is_fraud_node_train").mean().alias("community_fraud_ratio")
)
df_final = df_final.join(comm_fraud, on="community_id", how="left").fill_null(0)

# 최종 저장
df_final.write_parquet(output_path)

print("="*80)
print(f"🎉 [성공] 누락 방지 버전 그래프 피처 생성 완료!")
print(f"📁 저장 파일명: {output_path}")
print("="*80)

✅ cuGraph (GPU) 활성화


✅ NetworKit 활성화
📂 데이터 로딩 중...
✅ Timestamp 기준 정렬 완료
🔍 사기 계좌 추출 중...

🕸️ 그래프 알고리즘 계산 중...
📊 Clustering Coefficient 계산 중...


NetworKit Build: 100%|████████████████████████████████████████████| 3991748/3991748 [00:07<00:00, 504673.37it/s]



💾 결과 통합 중...


Finalizing: 100%|█████████████████████████████████████████████████| 2076999/2076999 [00:04<00:00, 508250.69it/s]


🎉 [성공] 누락 방지 버전 그래프 피처 생성 완료!
📁 저장 파일명: ./graph_features_v3_fixed_no_leakage.parquet


In [2]:
import polars as pl

# 1. 파일 로드
output_path = "./graph_features_v3_fixed_no_leakage.parquet"
df_check = pl.read_parquet(output_path)

print("==================================================")
print("✅ 1. 피처 존재 여부 및 데이터 타입 확인")
print("==================================================")
print(df_check.schema)

print("\n==================================================")
print("✅ 2. 주요 피처 통계량 (0 또는 Null 확인)")
print("==================================================")

# 리스트 내포를 사용하여 모든 메트릭에 대해 표현식을 생성하도록 수정
metrics = ["clustering_coefficient", "aml_suspicion_score", "community_fraud_ratio", "pagerank"]

# 각 컬럼별로 5개의 통계 지표를 생성합니다.
stats_exprs = []
for c in metrics:
    stats_exprs.extend([
        pl.col(c).count().alias(f"{c}_count"),
        pl.col(c).null_count().alias(f"{c}_nulls"),
        pl.col(c).mean().alias(f"{c}_mean"),
        pl.col(c).max().alias(f"{c}_max"),
        (pl.col(c) == 0).sum().alias(f"{c}_zeros")
    ])

stats = df_check.select(stats_exprs)
print(stats.glimpse())

print("\n==================================================")
print("✅ 3. 데이터 샘플 (상위 5행)")
print("==================================================")
# 보기 편하게 주요 컬럼들만 출력
print(df_check.select(["account_id"] + metrics).head(5))

# 4. 검증 로직
cc_max = df_check["clustering_coefficient"].max()
score_max = df_check["aml_suspicion_score"].max()

print("\n" + "="*50)
if cc_max > 0 and score_max > 0:
    print(f"✨ [검증 통과] \n - Clustering Max: {cc_max:.6f}\n - Suspicion Score Max: {score_max:.6f}")
    print("데이터가 정상적으로 들어있습니다. 이제 모델 학습을 진행하셔도 좋습니다!")
else:
    print("⚠️ [검증 실패] 특정 핵심 피처가 여전히 0입니다. 이전 단계의 계산 로직을 다시 확인해야 합니다.")
print("="*50)

✅ 1. 피처 존재 여부 및 데이터 타입 확인
Schema({'account_id': String, 'pagerank': Float64, 'in_degree': Int64, 'out_degree': Int64, 'clustering_coefficient': Float64, 'community_id': Int64, 'betweenness_centrality': Float64, 'degree_ratio': Float64, 'aml_suspicion_score': Float64, 'is_fraud_node_train': Int8, 'community_fraud_ratio': Float64})

✅ 2. 주요 피처 통계량 (0 또는 Null 확인)
Rows: 1
Columns: 20
$ clustering_coefficient_count <u32> 2076999
$ clustering_coefficient_nulls <u32> 0
$ clustering_coefficient_mean  <f64> 0.009219224732027617
$ clustering_coefficient_max   <f64> 1.0
$ clustering_coefficient_zeros <u32> 1961244
$ aml_suspicion_score_count    <u32> 2076999
$ aml_suspicion_score_nulls    <u32> 0
$ aml_suspicion_score_mean     <f64> 2.8871121775709246e-06
$ aml_suspicion_score_max      <f64> 0.002136711246262272
$ aml_suspicion_score_zeros    <u32> 1961244
$ community_fraud_ratio_count  <u32> 2076999
$ community_fraud_ratio_nulls  <u32> 0
$ community_fraud_ratio_mean   <f64> 0.009850269547553947


📋 실제 파일에 담긴 컬럼 (11개)
가장 처음에 나온 Schema 부분을 보시면 파일에 저장된 전체 목록이 나옵니다.

ID: account_id (1개)

그래프 알고리즘 피처: pagerank, in_degree, out_degree, clustering_coefficient, community_id, betweenness_centrality (6개)

파생/계산 피처: degree_ratio, aml_suspicion_score, is_fraud_node_train, community_fraud_ratio (4개)

총합: 11개